## Predictive Analytics final project ##

# Question 1 – Regression

## 1.1 Objective and Dataset Selection

The objective of this analysis is to predict an individual's body fat percentage using physical body measurements.

The `Body_fat.csv` dataset was selected because the target variable, `BodyFat`, is numerical and continuous, making it suitable for a regression task. The predictor variables include weight and measurements of the chest, abdomen, hip, thigh and biceps.

Two regression algorithms will be compared:

1. Linear Regression
2. Random Forest Regression

The models will be evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE) and the coefficient of determination (R²).

### Importing Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1.2 Data Loading

The dataset was imported into a Pandas DataFrame. The first five observations were displayed to examine its structure, variables and values.

In [3]:
df = pd.read_csv("data/Body_fat.csv")

df.head()

,Day,Gender,Weight,Chest,Abdomen,Hip,Thigh,Biceps,BodyFat
0,Monday,Male,154.25,93.1,85.2,94.5,59.0,32.0,12.3
1,Tuesday,Female,173.25,93.6,83.0,98.7,58.7,30.5,6.1
2,Wednesday,Female,154.00,95.8,87.9,99.2,59.6,28.8,25.3
3,Thursday,Male,184.75,101.8,86.4,101.2,60.1,32.4,10.4
4,Friday,Female,184.25,97.3,100.0,101.9,63.2,32.2,28.7


## 1.3 Data Cleaning and Initial Inspection

Before performing exploratory analysis and modelling, the dataset was examined for missing values, duplicated observations, incorrect data types and inconsistent categorical values.

Data cleaning does not always require deleting or modifying observations. In this analysis, the purpose of the cleaning stage is to verify the quality of the data and identify variables that should be removed or transformed before modelling.


In [11]:
# Check whether column names contain unnecessary spaces
column_name_changes = sum(
    df.columns != df.columns.str.strip()
)

# Check whether categorical values require standardisation
gender_changes = sum(
    df["Gender"] != df["Gender"].str.strip().str.title()
)

day_changes = sum(
    df["Day"] != df["Day"].str.strip().str.title()
)

print("Column names requiring correction:", column_name_changes)
print("Gender values requiring correction:", gender_changes)
print("Day values requiring correction:", day_changes)

# Apply standardisation
df.columns = df.columns.str.strip()
df["Gender"] = df["Gender"].str.strip().str.title()
df["Day"] = df["Day"].str.strip().str.title()

Column names requiring correction: 0
Gender values requiring correction: 0
Day values requiring correction: 0


### Data Quality Checks

The dataset was examined to determine its dimensions, data types, missing values, duplicated observations and the distribution of its categorical variables.

In [12]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nDataset information:")
df.info()

Number of rows: 251
Number of columns: 9

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Day      251 non-null    str    
 1   Gender   251 non-null    str    
 2   Weight   251 non-null    float64
 3   Chest    251 non-null    float64
 4   Abdomen  251 non-null    float64
 5   Hip      251 non-null    float64
 6   Thigh    251 non-null    float64
 7   Biceps   251 non-null    float64
 8   BodyFat  251 non-null    float64
dtypes: float64(7), str(2)
memory usage: 17.8 KB


In [13]:
# Check for missing values
missing_values = df.isnull().sum()

print("Missing values:")
display(missing_values.to_frame(name="Missing Values"))

# Check for duplicated observations
duplicate_count = df.duplicated().sum()

print("Number of duplicated rows:", duplicate_count)

Missing values:


,Missing Values
Day,0
Gender,0
Weight,0
Chest,0
Abdomen,0
Hip,0
Thigh,0
Biceps,0
BodyFat,0


Number of duplicated rows: 0


In [14]:
print("Gender values before formatting:")
print(df["Gender"].value_counts(dropna=False))

print("\nDay values before formatting:")
print(df["Day"].value_counts(dropna=False))

Gender values before formatting:
Gender
Female    130
Male      121
Name: count, dtype: int64

Day values before formatting:
Day
Thursday     73
Saturday     68
Friday       62
Monday       16
Tuesday      16
Wednesday    16
Name: count, dtype: int64


### Categorical Data Formatting

The column names and categorical variables were checked for unnecessary spaces and inconsistent capitalisation.

The column names were standardised by removing leading and trailing spaces. The `Gender` and `Day` variables were also standardised by removing unnecessary spaces and applying consistent capitalisation.

These transformations help prevent categories such as `Male`, `male` and ` Male` from being interpreted as different values during preprocessing.

In [16]:
# Remove possible spaces from column names
df.columns = df.columns.str.strip()

# Standardise the formatting of categorical variables
df["Gender"] = df["Gender"].str.strip().str.title()
df["Day"] = df["Day"].str.strip().str.title()

In [17]:
print("Gender values after formatting:")
print(df["Gender"].value_counts(dropna=False))

print("\nDay values after formatting:")
print(df["Day"].value_counts(dropna=False))

Gender values after formatting:
Gender
Female    130
Male      121
Name: count, dtype: int64

Day values after formatting:
Day
Thursday     73
Saturday     68
Friday       62
Monday       16
Tuesday      16
Wednesday    16
Name: count, dtype: int64


In [18]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Weight,251.0,178.190438,27.034801,118.5,158.75,176.25,196.875,262.75
Chest,251.0,100.683267,8.144414,79.3,94.30,99.60,105.300,128.30
Abdomen,251.0,92.334661,10.215190,69.4,84.55,90.90,99.200,126.20
Hip,251.0,99.714343,6.508078,85.0,95.50,99.30,103.350,125.60
Thigh,251.0,59.294821,4.954547,47.2,56.00,59.00,62.300,74.40
Biceps,251.0,32.222709,2.917904,24.8,30.20,32.00,34.300,39.10
BodyFat,251.0,19.094422,8.338962,0.0,12.45,19.20,25.250,47.50


### Data Quality Findings

The dataset contains 251 observations and 9 variables. The numerical body measurements and the target variable were stored using appropriate numerical data types, while `Day` and `Gender` were stored as categorical variables.

No missing values or duplicated observations were identified. The categorical variables were also consistently formatted, with no evidence of spelling errors, unnecessary spaces or inconsistent capitalisation.

As no missing, duplicated or clearly corrupted observations were found, no rows were removed during the initial cleaning process. Potential unusual numerical values will be investigated separately through outlier analysis.

## 1.4 Outlier Assessment